# Dynamic Model Fusion / Dynamic Classifier Selection

Notebook nay trien khai DMF/DCS cho hai baseline LSTM va GAT. Neu hai classifier dong thuan thi lay nhan chung; neu bat dong thi tinh score cho tung classifier bang validation competence, validation weight va confidence tren mau test, sau do chon classifier co score cao hon.

Input bat buoc:
- `/kaggle/input/datasets/tailength/dmf-artifacts/dmf_gat_lstm/lstm_val_predictions.csv`
- `/kaggle/input/datasets/tailength/dmf-artifacts/dmf_gat_lstm/lstm_test_predictions.csv`
- `/kaggle/input/datasets/tailength/dmf-artifacts/dmf_gat_lstm/gat_val_predictions.csv`
- `/kaggle/input/datasets/tailength/dmf-artifacts/dmf_gat_lstm/gat_test_predictions.csv`
- `label_mapping.csv` neu co.

Output duoc ghi vao `/kaggle/working/credit_rating_artifacts/dmf_gat_lstm` de tranh ghi vao Kaggle Dataset read-only.

## Notebook Contract
Notebook nay hop nhat LSTM va GAT bang DMF/DCS voi contract CSV co `row_id`.

- Input Kaggle read-only: `/kaggle/input/datasets/tailength/dmf-artifacts/dmf_gat_lstm` khi ton tai.
- Output writable: `/kaggle/working/credit_rating_artifacts/dmf_gat_lstm` tren Kaggle.
- Bat buoc co `lstm_*_predictions.csv`, `gat_*_predictions.csv` va nen co `label_mapping.csv`.
- Khong ghi vao `/kaggle/input`; moi metric/prediction moi phai ghi vao thu muc output.

## 1. Moi truong va Artifact Path

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, cohen_kappa_score, confusion_matrix, classification_report,
)
from sklearn.preprocessing import label_binarize

SEED = 42
np.random.seed(SEED)


def detect_kaggle_runtime() -> bool:
    if os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '').strip():
        return True
    return Path('/kaggle/input').exists() and Path('/kaggle/working').exists()


IN_KAGGLE = detect_kaggle_runtime()


def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'data').exists() and (p / 'src').exists():
            return p
    return start


PROJECT_ROOT = Path('/kaggle/working') if IN_KAGGLE else find_project_root(Path.cwd().resolve())
ARTIFACT_DIR = PROJECT_ROOT / 'credit_rating_artifacts'
KAGGLE_DMF_INPUT_ARTIFACT_DIR = Path('/kaggle/input/datasets/tailength/dmf-artifacts/dmf_gat_lstm')
DMF_ARTIFACT_DIR = ARTIFACT_DIR / 'dmf_gat_lstm'
DMF_INPUT_ARTIFACT_DIR = KAGGLE_DMF_INPUT_ARTIFACT_DIR if IN_KAGGLE and KAGGLE_DMF_INPUT_ARTIFACT_DIR.exists() else DMF_ARTIFACT_DIR
DMF_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)
print('DMF input artifact dir:', DMF_INPUT_ARTIFACT_DIR)
print('DMF output artifact dir:', DMF_ARTIFACT_DIR)


## 2. Load LSTM/GAT Prediction CSV

In [ ]:
LSTM_VAL_PATH = DMF_INPUT_ARTIFACT_DIR / 'lstm_val_predictions.csv'
LSTM_TEST_PATH = DMF_INPUT_ARTIFACT_DIR / 'lstm_test_predictions.csv'
GAT_VAL_PATH = DMF_INPUT_ARTIFACT_DIR / 'gat_val_predictions.csv'
GAT_TEST_PATH = DMF_INPUT_ARTIFACT_DIR / 'gat_test_predictions.csv'
LABEL_MAP_PATH = DMF_INPUT_ARTIFACT_DIR / 'label_mapping.csv'

required_paths = [LSTM_VAL_PATH, LSTM_TEST_PATH, GAT_VAL_PATH, GAT_TEST_PATH]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError('Thieu prediction CSV. Hay chay notebook LSTM va GAT truoc: ' + '; '.join(missing))

lstm_val_raw = pd.read_csv(LSTM_VAL_PATH, encoding='utf-8')
lstm_test_raw = pd.read_csv(LSTM_TEST_PATH, encoding='utf-8')
gat_val_raw = pd.read_csv(GAT_VAL_PATH, encoding='utf-8')
gat_test_raw = pd.read_csv(GAT_TEST_PATH, encoding='utf-8')
label_map = pd.read_csv(LABEL_MAP_PATH, encoding='utf-8') if LABEL_MAP_PATH.exists() else None

print('LSTM val/test:', lstm_val_raw.shape, lstm_test_raw.shape)
print('GAT  val/test:', gat_val_raw.shape, gat_test_raw.shape)
if label_map is not None:
    display(label_map)


## 3. Kiem tra Contract va Align Row

In [ ]:
def prob_columns(frame):
    cols = [c for c in frame.columns if c.startswith('prob_')]
    return sorted(cols, key=lambda x: int(x.split('_')[1]))


def standardize_predictions(frame, model_key):
    frame = frame.copy()
    probs = prob_columns(frame)
    required = {'row_id', 'true_label', 'pred_label', *probs}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f'{model_key} prediction file missing columns: {sorted(missing)}')
    keep_meta = [c for c in ['row_id', 'split', 'ticker', 'company_name', 'rating_date', 'true_label', 'true_label_name'] if c in frame.columns]
    out = frame[keep_meta].copy()
    out[f'{model_key}_pred_label'] = frame['pred_label'].astype(int)
    out[f'{model_key}_confidence'] = frame['confidence'].astype(float) if 'confidence' in frame.columns else frame[probs].max(axis=1).astype(float)
    for c in probs:
        out[f'{model_key}_{c}'] = frame[c].astype(float)
    return out


def align_pair(lstm_frame, gat_frame, split_name):
    lstm = standardize_predictions(lstm_frame, 'lstm')
    gat = standardize_predictions(gat_frame, 'gat')
    merged = lstm.merge(gat, on='row_id', suffixes=('_lstm_meta', '_gat_meta'), how='inner')
    if len(merged) != len(lstm) or len(merged) != len(gat):
        raise ValueError(
            f'Row_id mismatch on {split_name}: merged={len(merged)}, lstm={len(lstm)}, gat={len(gat)}. '
            'Kiem tra hai notebook co dung cung split va row_id khong.'
        )
    if not (merged['true_label_lstm_meta'].astype(int).values == merged['true_label_gat_meta'].astype(int).values).all():
        bad = merged.loc[merged['true_label_lstm_meta'].astype(int) != merged['true_label_gat_meta'].astype(int), 'row_id'].head().tolist()
        raise ValueError(f'True label mismatch on {split_name}; examples: {bad}')

    # Normalize metadata column names after the safety checks.
    merged['true_label'] = merged['true_label_lstm_meta'].astype(int)
    if 'true_label_name_lstm_meta' in merged.columns:
        merged['true_label_name'] = merged['true_label_name_lstm_meta']
    for col in ['split', 'ticker', 'company_name', 'rating_date']:
        a = f'{col}_lstm_meta'
        b = f'{col}_gat_meta'
        if a in merged.columns:
            merged[col] = merged[a]
        elif b in merged.columns:
            merged[col] = merged[b]
    return merged


val_df = align_pair(lstm_val_raw, gat_val_raw, 'val')
test_df = align_pair(lstm_test_raw, gat_test_raw, 'test')

n_classes = len([c for c in val_df.columns if c.startswith('lstm_prob_')])
class_ids = list(range(n_classes))
if label_map is not None and {'label_id', 'label_name'}.issubset(label_map.columns):
    id_to_name = {int(r.label_id): str(r.label_name) for r in label_map.itertuples(index=False)}
else:
    id_to_name = {i: str(i) for i in class_ids}
class_names = [id_to_name.get(i, str(i)) for i in class_ids]

print('Aligned val/test:', val_df.shape, test_df.shape, '| n_classes:', n_classes)
print('Agreement on test:', float((test_df['lstm_pred_label'] == test_df['gat_pred_label']).mean()))
display(test_df[['row_id', 'true_label', 'lstm_pred_label', 'gat_pred_label']].head())


## 4. Chuan hoa Probability Matrix

In [ ]:
def model_prob_matrix(frame, model_key):
    cols = [f'{model_key}_prob_{i}' for i in class_ids]
    missing = [c for c in cols if c not in frame.columns]
    if missing:
        raise ValueError(f'Missing probability columns for {model_key}: {missing}')
    probs = frame[cols].to_numpy(dtype=float)
    probs = np.clip(probs, 1e-9, 1.0)
    return probs / probs.sum(axis=1, keepdims=True)


def compute_metrics(y_true, y_pred, proba=None):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    out = {
        'Accuracy': float(accuracy_score(y_true, y_pred)),
        'Precision_Weighted': float(precision_score(y_true, y_pred, average='weighted', zero_division=0)),
        'Recall_Weighted': float(recall_score(y_true, y_pred, average='weighted', zero_division=0)),
        'Macro_F1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'Weighted_F1': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
        'QWK': float(cohen_kappa_score(y_true, y_pred, weights='quadratic')),
        'Ordinal_MAE': float(np.mean(np.abs(y_true - y_pred))),
    }
    if proba is not None:
        try:
            y_bin = label_binarize(y_true, classes=class_ids)
            out['AUC'] = float(roc_auc_score(y_bin, proba, average='macro', multi_class='ovr'))
        except Exception:
            out['AUC'] = float('nan')
    else:
        out['AUC'] = float('nan')
    return out


val_lstm_probs = model_prob_matrix(val_df, 'lstm')
val_gat_probs = model_prob_matrix(val_df, 'gat')
test_lstm_probs = model_prob_matrix(test_df, 'lstm')
test_gat_probs = model_prob_matrix(test_df, 'gat')
y_val = val_df['true_label'].to_numpy(dtype=int)
y_test = test_df['true_label'].to_numpy(dtype=int)

val_lstm_pred = val_lstm_probs.argmax(axis=1)
val_gat_pred = val_gat_probs.argmax(axis=1)
test_lstm_pred = test_lstm_probs.argmax(axis=1)
test_gat_pred = test_gat_probs.argmax(axis=1)

val_weights = {
    'lstm': accuracy_score(y_val, val_lstm_pred),
    'gat': accuracy_score(y_val, val_gat_pred),
}
print('Validation model weights v_j:', val_weights)


## DCS Scoring Rule

Voi moi mau test `x_t`:

1. Lay nhan du doan cua LSTM va GAT.
2. Neu hai nhan giong nhau, gan truc tiep nhan do.
3. Neu hai nhan khac nhau, voi tung model `j`:
   - `o_j`: nhan model `j` du doan cho `x_t`.
   - `delta_j`: trung binh top-k validation scores cua model `j` doi voi class `o_j`.
   - `v_j`: trong so global cua model `j` tren validation, mac dinh la validation accuracy.
   - `Z_j(x_t)`: confidence cua model `j` cho class `o_j` tren mau test.
   - `S_j = delta_j * v_j * Z_j(x_t)`.
4. Chon nhan cua classifier co `S_j` cao hon.

Cach nay giu fusion o muc row-level, nen prediction CSV phai co `row_id` va probability columns dong bo giua LSTM/GAT.

In [ ]:
DCS_TOP_K = 80


def topk_validation_competence(val_probs, predicted_class, k=DCS_TOP_K):
    scores = np.asarray(val_probs[:, int(predicted_class)], dtype=float)
    if len(scores) == 0:
        return 0.0
    k_eff = min(int(k), len(scores))
    if k_eff <= 0:
        return 0.0
    top = np.partition(scores, -k_eff)[-k_eff:]
    return float(np.mean(top))


def dcs_fuse_row(i):
    lstm_label = int(test_lstm_pred[i])
    gat_label = int(test_gat_pred[i])
    if lstm_label == gat_label:
        return {
            'final_label': lstm_label,
            'selected_model': 'agreement',
            'dcs_case': 'agreement',
            'lstm_delta': np.nan,
            'gat_delta': np.nan,
            'lstm_score': np.nan,
            'gat_score': np.nan,
        }

    lstm_delta = topk_validation_competence(val_lstm_probs, lstm_label, DCS_TOP_K)
    gat_delta = topk_validation_competence(val_gat_probs, gat_label, DCS_TOP_K)
    lstm_test_score = float(test_lstm_probs[i, lstm_label])
    gat_test_score = float(test_gat_probs[i, gat_label])
    lstm_score = lstm_delta * float(val_weights['lstm']) + lstm_test_score
    gat_score = gat_delta * float(val_weights['gat']) + gat_test_score

    if lstm_score >= gat_score:
        final_label = lstm_label
        selected_model = 'lstm'
    else:
        final_label = gat_label
        selected_model = 'gat'

    return {
        'final_label': final_label,
        'selected_model': selected_model,
        'dcs_case': 'disagreement',
        'lstm_delta': lstm_delta,
        'gat_delta': gat_delta,
        'lstm_score': lstm_score,
        'gat_score': gat_score,
    }


dcs_rows = [dcs_fuse_row(i) for i in range(len(test_df))]
dcs_df = pd.DataFrame(dcs_rows)
dmf_pred = dcs_df['final_label'].to_numpy(dtype=int)

# The selected classifier gives the final probability vector in DCS; agreement uses average probs for reporting AUC.
dmf_probs = np.zeros_like(test_lstm_probs)
for i, selected in enumerate(dcs_df['selected_model']):
    if selected == 'lstm':
        dmf_probs[i] = test_lstm_probs[i]
    elif selected == 'gat':
        dmf_probs[i] = test_gat_probs[i]
    else:
        dmf_probs[i] = (test_lstm_probs[i] + test_gat_probs[i]) / 2.0

dcs_summary = dcs_df['dcs_case'].value_counts().rename_axis('case').reset_index(name='count')
dcs_summary['ratio'] = dcs_summary['count'] / len(dcs_df)
display(dcs_summary)
print('DCS_TOP_K:', DCS_TOP_K)


## 6. Soft Voting Baseline

In [ ]:
soft_probs = (test_lstm_probs + test_gat_probs) / 2.0
soft_pred = soft_probs.argmax(axis=1)

weight_sum = float(val_weights['lstm'] + val_weights['gat'])
weighted_probs = (val_weights['lstm'] * test_lstm_probs + val_weights['gat'] * test_gat_probs) / max(weight_sum, 1e-12)
weighted_pred = weighted_probs.argmax(axis=1)

results = pd.DataFrame([
    {'Model': 'LSTM only', **compute_metrics(y_test, test_lstm_pred, test_lstm_probs)},
    {'Model': 'GAT only', **compute_metrics(y_test, test_gat_pred, test_gat_probs)},
    {'Model': 'Soft voting', **compute_metrics(y_test, soft_pred, soft_probs)},
    {'Model': 'Validation-weighted voting', **compute_metrics(y_test, weighted_pred, weighted_probs)},
    {'Model': 'DMF/DCS proposed', **compute_metrics(y_test, dmf_pred, dmf_probs)},
])

display(results)
metrics_path = DMF_ARTIFACT_DIR / 'dmf_dcs_metrics.csv'
results.to_csv(metrics_path, index=False, encoding='utf-8-sig')
print('Saved:', metrics_path)


## 7. Xuat Metric DMF/DCS

In [ ]:
out = test_df[['row_id', 'split', 'ticker', 'company_name', 'rating_date', 'true_label']].copy()
out['true_label_name'] = [id_to_name.get(int(y), str(y)) for y in out['true_label']]
out['lstm_pred_label'] = test_lstm_pred
out['lstm_pred_label_name'] = [id_to_name.get(int(y), str(y)) for y in test_lstm_pred]
out['lstm_confidence'] = test_lstm_probs.max(axis=1)
out['gat_pred_label'] = test_gat_pred
out['gat_pred_label_name'] = [id_to_name.get(int(y), str(y)) for y in test_gat_pred]
out['gat_confidence'] = test_gat_probs.max(axis=1)
out['dmf_pred_label'] = dmf_pred
out['dmf_pred_label_name'] = [id_to_name.get(int(y), str(y)) for y in dmf_pred]
out['dmf_confidence'] = dmf_probs.max(axis=1)
out = pd.concat([out.reset_index(drop=True), dcs_df.reset_index(drop=True)], axis=1)
for cls_idx in class_ids:
    out[f'lstm_prob_{cls_idx}'] = test_lstm_probs[:, cls_idx]
    out[f'gat_prob_{cls_idx}'] = test_gat_probs[:, cls_idx]
    out[f'dmf_prob_{cls_idx}'] = dmf_probs[:, cls_idx]

prediction_path = DMF_ARTIFACT_DIR / 'dmf_dcs_test_predictions.csv'
out.to_csv(prediction_path, index=False, encoding='utf-8-sig')
print('Saved:', prediction_path)
display(out.head())


## 8. Xuat Prediction CSV

In [ ]:
cm = confusion_matrix(y_test, dmf_pred, labels=class_ids)
cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)

plt.figure(figsize=(6, 5), dpi=160)
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('DMF/DCS Confusion Matrix (Test)')
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.tight_layout()
cm_plot_path = DMF_ARTIFACT_DIR / 'dmf_dcs_test_confusion_matrix.png'
plt.savefig(cm_plot_path, dpi=300, bbox_inches='tight')
plt.show()

display(cm_df)
print('Classification report (DMF/DCS test set):')
print(classification_report(
    y_test,
    dmf_pred,
    labels=class_ids,
    target_names=class_names,
    digits=4,
    zero_division=0,
))

cls_report_df = pd.DataFrame(
    classification_report(
        y_test,
        dmf_pred,
        labels=class_ids,
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )
).transpose()
cm_csv_path = DMF_ARTIFACT_DIR / 'dmf_dcs_test_confusion_matrix.csv'
cls_csv_path = DMF_ARTIFACT_DIR / 'dmf_dcs_test_classification_report.csv'
cm_df.to_csv(cm_csv_path, encoding='utf-8-sig')
cls_report_df.to_csv(cls_csv_path, encoding='utf-8-sig')
print('Saved:', cm_plot_path)
print('Saved:', cm_csv_path)
print('Saved:', cls_csv_path)


## 9. Confusion Matrix va Report

In [ ]:
# Inspect disagreement cases where DCS actually selected between LSTM and GAT.
disagreements = out[out['dcs_case'].eq('disagreement')].copy()
if len(disagreements):
    preview_cols = [
        'row_id', 'ticker', 'rating_date', 'true_label_name',
        'lstm_pred_label_name', 'gat_pred_label_name', 'dmf_pred_label_name',
        'selected_model', 'lstm_score', 'gat_score',
    ]
    display(disagreements[preview_cols].head(20))
else:
    print('Khong co case bat dong giua LSTM va GAT tren test set.')
